# Lähde-ekstraktion pohja

**MIKSI tämä notebook on olemassa:**
Yksi tutkimusartikkeli = yksi notebook = yksi `data/interim/{first_author}_{year}.csv` -tiedosto.
Manuaalisesti syötetyt rivit tarkistetaan skeemavalidoinnilla ja SMILES-kanonisoinnilla
ennen tallennusta. Tämä takaa, että jokainen lähde lisätään korpukseen samalla tavalla.

**Käyttöohje:**
1. Kopioi tämä tiedosto nimelle `{first_author}_{year}.ipynb` (esim. `lobmann_2013.ipynb`).
2. Täytä bibliografiset tiedot ja DOI.
3. Syötä rivit `entries`-listaan dictinä per pari.
4. Aja kaikki solut. Jos validointi epäonnistuu, korjaa rivit ja aja uudelleen.
5. Tarkista, että `data/interim/{first_author}_{year}.csv` syntyi oikein.
6. Aja `merge.merge_interim(...)` tai päivitä master-CSV erikseen H1:n päätyttyä.

## (a) Bibliografiset tiedot

In [ ]:
# Täytä tämän lähteen bibliografia. Nämä menevät jokaiseen riviin source_*-sarakkeisiin.
SOURCE = {
    "source_doi": "10.XXXX/example",
    "source_first_author": "Author",
    "source_year": 2024,
    "source_table_or_figure": "Table 2",
    "extraction_date": "2026-05-04",
    "extracted_by": "PH",
}

FIRST_AUTHOR_SLUG = SOURCE["source_first_author"].lower()
OUTPUT_FILENAME = f"{FIRST_AUTHOR_SLUG}_{SOURCE['source_year']}.csv"

## (b) Manuaalinen datan syöttö

Yksi `dict` per (drug_A, drug_B) -pari. Jätä puuttuvat arvot `None`:ksi (älä keksi).
Sarakkeet voi tarkistaa `configs/corpus_schema.yaml`:sta.

In [ ]:
entries = [
    # Esimerkki (poista ennen oikeaa ekstraktiota):
    # {
    #     "entry_id": "Table2_row1",
    #     "drug_A_name": "Indometacin",
    #     "drug_B_name": "Naproxen",
    #     "drug_A_role": "api_api",
    #     "drug_B_role": "api_api",
    #     "drug_A_smiles_original": "CC1=C(C2=CC(=CC=C2N1C(=O)C3=CC=C(C=C3)Cl)OC)CC(=O)O",
    #     "drug_B_smiles_original": "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
    #     "mole_fraction_A": 0.5,
    #     "mole_fraction_B": 0.5,
    #     "ratio_reported_as": "mole_fraction",
    #     "process_method": "melt_quench",
    #     "induction_time_days": 200.0,
    #     "induction_time_censored": False,
    #     "storage_T_C": 40.0,
    #     "storage_RH_percent": 75.0,
    #     "Tg_K": 333.0,
    #     "pxrd_amorphous": True,
    #     "detection_methods": "PXRD,DSC",
    # },
]

## (c) DataFrame-konstruktio

In [ ]:
import logging
from pathlib import Path

import pandas as pd

from coamorphous.corpus.canonicalize import canonical_smiles, inchikey_from_smiles, safe_canonical
from coamorphous.corpus.schema import build_schema, column_order, load_schema_yaml

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "configs" / "corpus_schema.yaml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Projektin juurta ei löydy (configs/corpus_schema.yaml puuttuu).")
    PROJECT_ROOT = PROJECT_ROOT.parent

SCHEMA_YAML = PROJECT_ROOT / "configs" / "corpus_schema.yaml"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

spec = load_schema_yaml(SCHEMA_YAML)
cols = column_order(spec)

# Lisätään SOURCE-tiedot ja pair_id jokaiselle riville.
rows = []
for i, e in enumerate(entries, start=1):
    row = {**SOURCE, **e}
    row["pair_id"] = f"{FIRST_AUTHOR_SLUG}{SOURCE['source_year']}_{i:02d}"
    rows.append(row)

df = pd.DataFrame(rows, columns=cols)
df.head()

## (d) Skeemavalidointi

In [ ]:
# Tässä vaiheessa Pandera tarkistaa tyypit ja enumit. Jos tämä epäonnistuu,
# korjaa entries-lista yllä ja aja solut uudelleen.
schema = build_schema(SCHEMA_YAML)

if df.empty:
    print("Ei riviä syötetty - skipataan validointi.")
else:
    df = schema.validate(df, lazy=True)
    print(f"OK: {len(df)} riviä läpäisi skeemavalidoinnin.")

## (e) SMILES-kanonisointi ja InChIKey-haku

In [ ]:
# MIKSI tässä: original-SMILES säilytetään auditointia varten,
# kanoninen muoto käytetään duplikaattitarkistuksiin ja deskriptorilaskentaan.
if not df.empty:
    df["drug_A_smiles_canonical"] = df["drug_A_smiles_original"].apply(safe_canonical)
    df["drug_B_smiles_canonical"] = df["drug_B_smiles_original"].apply(safe_canonical)
    df["drug_A_inchikey"] = df["drug_A_smiles_canonical"].apply(
        lambda s: inchikey_from_smiles(s) if s else None
    )
    df["drug_B_inchikey"] = df["drug_B_smiles_canonical"].apply(
        lambda s: inchikey_from_smiles(s) if s else None
    )
    # Validoidaan uudelleen, koska sarakkeet muuttuivat.
    df = schema.validate(df, lazy=True)
    df[["drug_A_name", "drug_A_smiles_canonical", "drug_A_inchikey"]].head()

## (f) Tallennus interim-tiedostoon

In [ ]:
output_path = INTERIM_DIR / OUTPUT_FILENAME
df.to_csv(output_path, index=False)
print(f"Tallennettu {len(df)} riviä -> {output_path}")